# Superstore SQL Analysis

Independent analysis for Assignment 3 only. The goal here is to load the Superstore CSV into SQLite, split it into a few cleaner tables, and then answer the business questions with subqueries, CTEs, and window functions.

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd
from IPython.display import display

base_path = Path('c:/Users/Subham Singhania/Desktop/celebal/Assignment-3')
csv_path = base_path / 'Sample - Superstore.csv'

df = pd.read_csv(csv_path)
df.columns = [
    col.strip().lower().replace(' ', '_').replace('-', '_')
    for col in df.columns
]

for col in ['order_date', 'ship_date']:
    df[col] = pd.to_datetime(df[col], errors='coerce').dt.strftime('%Y-%m-%d')

display(df.head())
print(f'rows: {len(df):,}')
print(f'columns: {len(df.columns):,}')

In [ ]:
conn = sqlite3.connect(':memory:')
df.to_sql('superstore_raw', conn, if_exists='replace', index=False)

conn.executescript('''
DROP TABLE IF EXISTS customers;
CREATE TABLE customers AS
SELECT DISTINCT
    customer_id,
    customer_name,
    segment,
    country,
    city,
    state,
    postal_code,
    region
FROM superstore_raw;

DROP TABLE IF EXISTS orders;
CREATE TABLE orders AS
SELECT DISTINCT
    order_id,
    order_date,
    ship_date,
    ship_mode,
    customer_id,
    segment,
    country,
    city,
    state,
    postal_code,
    region
FROM superstore_raw;

DROP TABLE IF EXISTS products;
CREATE TABLE products AS
SELECT DISTINCT
    product_id,
    category,
    sub_category,
    product_name
FROM superstore_raw;
''')

table_counts = pd.read_sql_query(
    '''
    SELECT 'superstore_raw' AS table_name, COUNT(*) AS row_count FROM superstore_raw
    UNION ALL
    SELECT 'customers', COUNT(*) FROM customers
    UNION ALL
    SELECT 'orders', COUNT(*) FROM orders
    UNION ALL
    SELECT 'products', COUNT(*) FROM products
    ''',
    conn,
)

display(table_counts)

In [ ]:
def q(sql: str) -> pd.DataFrame:
    return pd.read_sql_query(sql, conn)

above_average_sales = q('''
    SELECT
        order_id,
        customer_id,
        customer_name,
        product_id,
        product_name,
        sales,
        profit
    FROM superstore_raw
    WHERE sales > (SELECT AVG(sales) FROM superstore_raw)
    ORDER BY sales DESC
    LIMIT 10
''')

display(above_average_sales)

highest_order_per_customer = q('''
    WITH order_totals AS (
        SELECT
            o.customer_id,
            c.customer_name,
            o.order_id,
            SUM(r.sales) AS order_sales
        FROM superstore_raw r
        JOIN orders o ON r.order_id = o.order_id
        JOIN customers c ON o.customer_id = c.customer_id
        GROUP BY o.customer_id, c.customer_name, o.order_id
    )
    SELECT
        customer_id,
        customer_name,
        order_id,
        order_sales
    FROM order_totals ot
    WHERE order_sales = (
        SELECT MAX(order_sales)
        FROM order_totals
        WHERE customer_id = ot.customer_id
    )
    ORDER BY order_sales DESC
    LIMIT 10
''')

display(highest_order_per_customer)

In [ ]:
customer_rankings = q('''
    WITH customer_sales AS (
        SELECT
            c.customer_id,
            c.customer_name,
            c.segment,
            SUM(r.sales) AS total_sales,
            COUNT(DISTINCT o.order_id) AS total_orders
        FROM superstore_raw r
        JOIN customers c ON r.customer_id = c.customer_id
        JOIN orders o ON r.order_id = o.order_id
        GROUP BY c.customer_id, c.customer_name, c.segment
    ),
    ranked_customers AS (
        SELECT
            customer_id,
            customer_name,
            segment,
            total_sales,
            total_orders,
            ROW_NUMBER() OVER (ORDER BY total_sales DESC) AS row_number_rank,
            RANK() OVER (ORDER BY total_sales DESC) AS sales_rank
        FROM customer_sales
    )
    SELECT *
    FROM ranked_customers
    ORDER BY sales_rank, customer_name
''')

display(customer_rankings.head(10))
display(customer_rankings.tail(10))

single_order_customers = customer_rankings[customer_rankings['total_orders'] == 1].copy()
display(single_order_customers.sort_values('total_sales', ascending=False).head(10))

In [ ]:
top_customers = customer_rankings.head(10).copy()
low_customers = customer_rankings.sort_values(['total_sales', 'customer_name']).head(10).copy()
above_average_count = len(above_average_sales)
single_order_count = len(single_order_customers)

highest_customer = customer_rankings.iloc[0]
lowest_customer = customer_rankings.iloc[-1]

print('Top customers by total sales:')
print(top_customers[['customer_name', 'total_sales', 'total_orders']].to_string(index=False))
print()
print('Low customers by total sales:')
print(low_customers[['customer_name', 'total_sales', 'total_orders']].to_string(index=False))
print()
print('Business notes:')
print(f'- Line items above average sales in the first sample slice: {above_average_count}')
print(f'- Single-order customers found in the dataset: {single_order_count}')
print(f'- Highest ranked customer: {highest_customer["customer_name"]} with sales {highest_customer["total_sales"]:.2f}')
print(f'- Bottom customer in the ranking slice: {lowest_customer["customer_name"]} with sales {lowest_customer["total_sales"]:.2f}')

## Quick read

The interesting part is not just who sells the most, but how concentrated the sales are. A few customers usually carry a very large chunk of revenue, while a long tail of single-order customers sits near the bottom. That is the part I would actually act on first: keep the heavy buyers warm, and try to convert the one-time buyers into repeat customers.